# Ensemble Model Development

This notebook combines the shared preprocessing, the three tuned base models, and the ensemble evaluation.

The workflow uses training cross-validation to select the two base models and the ensemble method. The held-out test set is used only once, after those choices have been made, to compare the three base models with the selected ensemble.

**Data requirement:** Place the original `bank-full.csv` file in `data/raw/`. The raw dataset is excluded from GitHub.

## 1. Set up the project and load the shared functions

In [ ]:
from pathlib import Path
import sys
import json

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "src" / "shared_preprocessing.py").is_file():
    raise FileNotFoundError("Open this notebook from the repository or notebooks folder.")

sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import cross_val_score

from src.shared_preprocessing import (
    PRIMARY_SCORING,
    build_preprocessor,
    get_split,
    make_cv,
)
from src.ensemble import (
    choose_two_models,
    make_ensembles,
    evaluate_candidates,
)
from src.final_workflow_xinyi import zyen_runner
from src.evaluation_plots import (
    plot_model_comparison,
    plot_confusion_matrices,
    plot_roc_curves,
)

print("Project root:", ROOT)
print("Primary selection metric:", PRIMARY_SCORING)
print("Cross-validation folds:", make_cv().get_n_splits())

## 2. Load the shared training and test split

The split and feature preparation come from `src/shared_preprocessing.py`. The test data are kept aside while the base models and ensemble method are selected.

In [ ]:
X_train, X_test, y_train, y_test = get_split()

print(f"Training records: {len(X_train):,}")
print(f"Test records: {len(X_test):,}")
print(f"Input features before encoding: {X_train.shape[1]}")
print(f"Positive class in training: {y_train.mean():.2%}")
print(f"Positive class in test: {y_test.mean():.2%}")

preprocessor = build_preprocessor(X_train)

## 3. Train and tune the three base models

The model definitions and search spaces are taken from `notebooks/02_base_models.ipynb`. Grid search uses five-fold stratified cross-validation and F1-score. This step does not evaluate the test set.

In [ ]:
run_base_models = zyen_runner()

cv_results, tuned_models = run_base_models(
    preprocessor,
    X_train,
    y_train,
    scoring=PRIMARY_SCORING,
)

display(cv_results.round(4))

if len(tuned_models) != 3:
    raise RuntimeError("Expected exactly three tuned base models.")

## 4. Select two base models using training cross-validation

The two models with the highest tuned mean cross-validation F1-scores are selected. The remaining model is excluded from the ensemble, but will still be included in the final test comparison.

In [ ]:
cv_f1 = dict(zip(cv_results["model"], cv_results["tuned_cv_mean"]))
selected_models = choose_two_models(cv_f1)
excluded_models = [name for name in tuned_models if name not in selected_models]

print("Selected base models:", selected_models)
print("Excluded base model:", excluded_models[0])


## 5. Compare Soft Voting and Stacking using training cross-validation

Both ensemble methods are evaluated using the same five-fold stratified cross-validation strategy. The method with the higher mean F1-score is selected before the test set is used.

In [ ]:
ensemble_candidates = make_ensembles(tuned_models, selected_models)

ensemble_cv_scores = {}
for name, estimator in ensemble_candidates.items():
    scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        scoring=PRIMARY_SCORING,
        cv=make_cv(),
        n_jobs=1,
        error_score="raise",
    )
    ensemble_cv_scores[name] = float(np.mean(scores))

ensemble_cv_results = (
    pd.DataFrame(
        [{"Ensemble": name, "Mean CV F1": score}
         for name, score in ensemble_cv_scores.items()]
    )
    .sort_values("Mean CV F1", ascending=False)
    .reset_index(drop=True)
)

selected_ensemble = max(ensemble_cv_scores, key=ensemble_cv_scores.get)
display(ensemble_cv_results.round(4))
print("Selected ensemble:", selected_ensemble)

## 6. Final evaluation on the held-out test set

Model tuning, base-model selection and ensemble selection are now complete. The test set is used once for the final comparison of all three base models and the selected ensemble.

In [ ]:
test_results, fitted_models, confusion_matrices = evaluate_candidates(
    tuned_models,
    selected_models,
    X_train,
    y_train,
    X_test,
    y_test,
    ensemble_method=selected_ensemble,
)

display(test_results.round(4))

## 7. Confusion matrices and ROC curves

The confusion matrices show the correct and incorrect predictions for each model. The ROC curves compare how well the models separate subscribers from non-subscribers across different thresholds.

In [ ]:
confusion_figure = plot_confusion_matrices(confusion_matrices)
display(confusion_figure)

roc_figure = plot_roc_curves(fitted_models, X_test, y_test)
display(roc_figure)

## 8. Save the final results

The output files are saved under `results/xinyi_final/` so that the tables and figures used for the report can be traced to the experiment.

In [ ]:
OUTPUT_DIR = ROOT / "results" / "xinyi_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cv_results.to_csv(OUTPUT_DIR / "training_cv.csv", index=False)

base_test = test_results.loc[
    list(tuned_models.keys()),
    ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
].rename_axis("model").reset_index()
base_test.insert(1, "stage", "tuned")
base_test = base_test.rename(columns={
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC-AUC": "roc_auc",
})
base_test.to_csv(OUTPUT_DIR / "zyen_base_test.csv", index=False)
test_results.to_csv(OUTPUT_DIR / "test_comparison.csv")

notes = {
    "status": "Final integrated workflow",
    "shared_preprocessing": "src/shared_preprocessing.py",
    "base_model_definitions": "notebooks/02_base_models.ipynb",
    "cv_folds": make_cv().get_n_splits(),
    "selected_models": list(selected_models),
    "excluded_model": excluded_models[0],
    "ensemble_cv_f1": ensemble_cv_scores,
    "chosen_ensemble": selected_ensemble,
    "test_set_used_after_model_selection": True,
}
(OUTPUT_DIR / "experiment_notes.json").write_text(
    json.dumps(notes, indent=2),
    encoding="utf-8",
)

for filename, figure in (
    ("model_comparison.png", plot_model_comparison(test_results)),
    ("confusion_matrices.png", confusion_figure),
    ("roc_curves.png", roc_figure),
):
    figure.savefig(OUTPUT_DIR / filename, dpi=150, bbox_inches="tight")

print("Saved final results to:", OUTPUT_DIR)

## 9. Result summary

The final result files are:

- `training_cv.csv`: baseline and tuned cross-validation results for the three base models.
- `zyen_base_test.csv`: final test metrics for the three base models, taken from the single final evaluation.
- `test_comparison.csv`: final test metrics for all three base models and the selected ensemble.
- `experiment_notes.json`: selected models, selected ensemble and experiment settings.
- `model_comparison.png`, `confusion_matrices.png`, and `roc_curves.png`: evaluation figures.

The test metrics are not used to choose the base models or the ensemble method.